## A Spectabular Model of a Simple Elevator

A simple elevator serves three floors (F1, F2, F3) with four call buttons:
- **Floor 1** (ground): Up button
- **Floor 2** (middle): Up and Down buttons
- **Floor 3** (top): Down button

Seven events in the system
- pressing one of the 4 call buttons
- elevator arriving at one of the 3 floors

The specification is given as tabular expressions using the Spectabular library.

In [1]:
%run "spectabular.ipynb"

### State Variables

The elevator's floor is modelled as an enumeration type `Floor` with values `F1`, `F2`, `F3`. Four Boolean variables model the call buttons: floor 1 has only an Up button, floor 3 only a Down button, and floor 2 has both. A Boolean `doorOpen` tracks whether the elevator door is open.

In [2]:
EnumType('Floor', 'F1', 'F2', 'F3')
Floor('floor')

Bool('b1u')      # call button: floor 1, up
Bool('b2u')      # call button: floor 2, up
Bool('b2d')      # call button: floor 2, down
Bool('b3d')      # call button: floor 3, down
Bool('doorOpen') # elevator door is open

### Event Type

The seven events are modelled as an enumeration: four button presses and three floor arrivals.

In [3]:
Enum('ev', 'Press1U', 'Press2U', 'Press2D', 'Press3D', 'ArrF1', 'ArrF2', 'ArrF3')

### Invariant

When the elevator door is open at a floor, the call buttons at that floor must be inactive (the call has been served). This is expressed as three implications, one per floor:

In [4]:
INV1 = \
    Implies(doorOpen & (floor == F1), ~b1u); INV1

Implies(And(doorOpen, floor == F1), Not(b1u))

In [5]:
INV2 = \
    Implies(doorOpen & (floor == F2), ~b2u & ~b2d); INV2

Implies(And(doorOpen, floor == F2), And(Not(b2u), Not(b2d)))

In [6]:
INV3 = \
    Implies(doorOpen & (floor == F3), ~b3d); INV3

Implies(And(doorOpen, floor == F3), Not(b3d))

In [7]:
INV = INV1 & INV2 & INV3; INV

And(And(Implies(And(doorOpen, floor == F1), Not(b1u)),
        Implies(And(doorOpen, floor == F2),
                And(Not(b2u), Not(b2d)))),
    Implies(And(doorOpen, floor == F3), Not(b3d)))

### Button Press Events

When a call button is pressed, that button becomes active and the door closes. The floor and all other buttons are unchanged. Reading the table vertically: pressing button 1U sets `b1u` to true, closes the door, and preserves all other state. The same follows for the other buttons.

In [8]:
pressButtonEvent = \
    VectorTable(
        (ev == Press1U, ev == Press2U, ev == Press2D, ev == Press3D),
        (b1uʹ,      True,  b1u,   b1u,   b1u),
        (b2uʹ,      b2u,   True,  b2u,   b2u),
        (b2dʹ,      b2d,   b2d,   True,  b2d),
        (b3dʹ,      b3d,   b3d,   b3d,   True),
        (floorʹ,    floor, floor, floor, floor),
        (doorOpenʹ, False, False, False, False)); pressButtonEvent

The header is disjoint but not total:

In [24]:
assert pressButtonEvent.topdisjoint
assert not pressButtonEvent.toptotal

### Arrival Events

When the elevator arrives at a floor, the floor variable is updated, the call buttons at that floor are cleared, and the door opens. Buttons at other floors are preserved. Arriving at floor 2 clears both the up and down buttons.

In [25]:
arriveEvent = \
    VectorTable(
        (ev == ArrF1, ev == ArrF2, ev == ArrF3),
        (floorʹ,    F1,    F2,    F3),
        (b1uʹ,      False, b1u,   b1u),
        (b2uʹ,      b2u,   False, b2u),
        (b2dʹ,      b2d,   False, b2d),
        (b3dʹ,      b3d,   b3d,   False),
        (doorOpenʹ, True,  True,  True)); arriveEvent

The arrival header is also disjoint but not total:

In [26]:
assert arriveEvent.topdisjoint
assert not arriveEvent.toptotal

### Full Elevator Specification

The entire specification is written as a single vector table with all seven event columns. Reading horizontally confirms, for example, that `b1u` is set to true only by `Press1U` and cleared only by `ArrF1`, and that the door opens only on arrival.

In [28]:
elevatorSpec = \
    VectorTable(
        (ev == Press1U, ev == Press2U, ev == Press2D, ev == Press3D,
         ev == ArrF1,   ev == ArrF2,   ev == ArrF3),
        (floorʹ,    floor, floor, floor, floor, F1,    F2,    F3),
        (b1uʹ,      True,  b1u,   b1u,   b1u,  False, b1u,   b1u),
        (b2uʹ,      b2u,   True,  b2u,   b2u,  b2u,   False, b2u),
        (b2dʹ,      b2d,   b2d,   True,  b2d,  b2d,   False, b2d),
        (b3dʹ,      b3d,   b3d,   b3d,   True, b3d,   b3d,   False),
        (doorOpenʹ, False, False, False, False, True,  True,  True)); elevatorSpec

The combined header covers all seven events and is both total and disjoint, showing the specification is deterministic:

In [29]:
assert elevatorSpec.toptotal
assert elevatorSpec.topdisjoint

In [30]:
flatten(pressButtonEvent)

Or(And(ev == Press1U,
       And(b1uʹ == True,
           b2uʹ == b2u,
           b2dʹ == b2d,
           b3dʹ == b3d,
           floorʹ == floor,
           doorOpenʹ == False)),
   And(ev == Press2U,
       And(b1uʹ == b1u,
           b2uʹ == True,
           b2dʹ == b2d,
           b3dʹ == b3d,
           floorʹ == floor,
           doorOpenʹ == False)),
   And(ev == Press2D,
       And(b1uʹ == b1u,
           b2uʹ == b2u,
           b2dʹ == True,
           b3dʹ == b3d,
           floorʹ == floor,
           doorOpenʹ == False)),
   And(ev == Press3D,
       And(b1uʹ == b1u,
           b2uʹ == b2u,
           b2dʹ == b2d,
           b3dʹ == True,
           floorʹ == floor,
           doorOpenʹ == False)))

The flattened form of the arrival event table:

In [31]:
flatten(arriveEvent)

Or(And(ev == ArrF1,
       And(floorʹ == F1,
           b1uʹ == False,
           b2uʹ == b2u,
           b2dʹ == b2d,
           b3dʹ == b3d,
           doorOpenʹ == True)),
   And(ev == ArrF2,
       And(floorʹ == F2,
           b1uʹ == b1u,
           b2uʹ == False,
           b2dʹ == False,
           b3dʹ == b3d,
           doorOpenʹ == True)),
   And(ev == ArrF3,
       And(floorʹ == F3,
           b1uʹ == b1u,
           b2uʹ == b2u,
           b2dʹ == b2d,
           b3dʹ == False,
           doorOpenʹ == True)))

### Invariant Preservation

The invariant `INV` states that when the door is open at a floor, the call buttons at that floor are inactive. We verify that `INV` is preserved by each event, using the Hoare-style correctness triple `{INV} S {INV}`.

In [34]:
Correct("INV", elevatorSpec, "INV")

In [35]:
assert valid(_)

### Domain

The domain of the full specification covers all possible states.

In [37]:
Dom(elevatorSpec)

In [38]:
flatten(_)

Exists([b2dʹ, b3dʹ, doorOpenʹ, b1uʹ, b2uʹ, floorʹ],
       Or(And(ev == Press1U,
              And(floorʹ == floor,
                  b1uʹ == True,
                  b2uʹ == b2u,
                  b2dʹ == b2d,
                  b3dʹ == b3d,
                  doorOpenʹ == False)),
          And(ev == Press2U,
              And(floorʹ == floor,
                  b1uʹ == b1u,
                  b2uʹ == True,
                  b2dʹ == b2d,
                  b3dʹ == b3d,
                  doorOpenʹ == False)),
          And(ev == Press2D,
              And(floorʹ == floor,
                  b1uʹ == b1u,
                  b2uʹ == b2u,
                  b2dʹ == True,
                  b3dʹ == b3d,
                  doorOpenʹ == False)),
          And(ev == Press3D,
              And(floorʹ == floor,
                  b1uʹ == b1u,
                  b2uʹ == b2u,
                  b2dʹ == b2d,
                  b3dʹ == True,
                  doorOpenʹ == False)),
          And(ev == ArrF1,
              And(floorʹ == F1,
                  b1uʹ == False,
                  b2uʹ == b2u,
                  b2dʹ == b2d,
                  b3dʹ == b3d,
                  doorOpenʹ == True)),
          And(ev == ArrF2,
              And(floorʹ == F2,
                  b1uʹ == b1u,
                  b2uʹ == False,
                  b2dʹ == False,
                  b3dʹ == b3d,
                  doorOpenʹ == True)),
          And(ev == ArrF3,
              And(floorʹ == F3,
                  b1uʹ == b1u,
                  b2uʹ == b2u,
                  b2dʹ == b2d,
                  b3dʹ == False,
                  doorOpenʹ == True))))